# GCN+LSTM diagnostics

Analisa o impacto da componente GCN (e do threshold do grafo Spearman) para um produto seleccionado, a partir dos CSVs gerados pelo grid runner em `main.py`.

**Inputs esperados** (mesma pasta deste notebook):
- `diagnostics.csv` — uma linha por época para cada `(produto, seed, threshold, config)`.
- `spearman.csv` — uma linha por experiência com as métricas finais (RMSE, MAE, …).
- `diagnostics_ablation.csv` (opcional) — gerado quando `ABLATE_Z=True` em `main.py`.

**Checklist de diagnóstico**:
1. **Topologia** muda com o threshold? (`num_edges_mean` por threshold)
2. **Convergência** do treino — quantas épocas até `val_loss` estagnar?
3. **GCN viva ou morta?** `‖z‖`, `Var(z)` e `‖∇gcn‖/‖∇lstm‖` ao longo das épocas.
4. **O threshold importa?** Final RMSE vs threshold (média ± std sobre seeds).
5. **A GCN contribui?** Comparação baseline vs ablation (se disponível).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Configuration ────────────────────────────────────────────────────────
SCRIPT_DIR   = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
DIAG_PATH    = os.path.join(SCRIPT_DIR, "diagnostics.csv")
ABLAT_PATH   = os.path.join(SCRIPT_DIR, "diagnostics_ablation.csv")
RESULTS_PATH = os.path.join(SCRIPT_DIR, "spearman.csv")

# Choose which product to analyse
PRODUCT_ID = 26008      # change to whichever item is in diagnostics.csv
METRIC     = "spearman"

print("Reading from:", SCRIPT_DIR)
print("diagnostics.csv :", os.path.exists(DIAG_PATH))
print("ablation csv    :", os.path.exists(ABLAT_PATH))
print("results csv     :", os.path.exists(RESULTS_PATH))


In [ ]:
diag = pd.read_csv(DIAG_PATH)
diag = diag[(diag["product_id"] == PRODUCT_ID) & (diag["metric"] == METRIC)].copy()
diag["threshold"] = pd.to_numeric(diag["threshold"], errors="coerce")

print(f"Rows for product {PRODUCT_ID}: {len(diag)}")
print("Seeds      :", sorted(diag["seed"].unique()))
print("Thresholds :", sorted(diag["threshold"].dropna().unique()))
print("Max epoch  :", int(diag["epoch"].max()))
diag.head()


## 1. Topologia — o threshold muda mesmo o grafo?

`num_edges_mean` é constante para cada `(produto, threshold)` (o grafo é construído uma vez). Se os valores de `num_edges_mean` forem semelhantes entre thresholds, o threshold **não está a fazer nada** na vizinhança — e nenhuma análise posterior vai mostrar dependência.


In [ ]:
topo = (
    diag.dropna(subset=["threshold"])
        .groupby("threshold")["num_edges_mean"]
        .first()
        .reset_index()
        .sort_values("threshold")
)
print(topo.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(topo["threshold"], topo["num_edges_mean"], marker="o", lw=2)
ax.set_xlabel("Threshold"); ax.set_ylabel("Mean # edges per training window")
ax.set_title(f"Product {PRODUCT_ID} — graph density vs threshold")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 2. Convergência — quando estagna o `val_loss`?

`val_loss` ao longo das épocas, por seed e threshold. Se as curvas plateau-am todas no mesmo valor, o threshold é irrelevante para a optimização. Se algumas curvas atingem mínimos visivelmente menores, há sinal extraível.


In [ ]:
seeds      = sorted(diag["seed"].unique())
thresholds = sorted(diag["threshold"].dropna().unique())
cmap       = plt.get_cmap("viridis")
colors     = {t: cmap(i / max(len(thresholds) - 1, 1)) for i, t in enumerate(thresholds)}

fig, axes = plt.subplots(1, len(seeds), figsize=(5 * len(seeds), 4), sharey=True, squeeze=False)
for ax, seed in zip(axes[0], seeds):
    for t in thresholds:
        sub = diag[(diag["seed"] == seed) & (diag["threshold"] == t)].sort_values("epoch")
        if sub.empty:
            continue
        ax.plot(sub["epoch"], sub["val_loss"], color=colors[t], label=f"th={t}", lw=1.2)
    ax.set_title(f"Seed {seed}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Val loss"); ax.set_yscale("log")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle(f"Product {PRODUCT_ID} — val_loss per epoch", y=1.02)
plt.tight_layout(); plt.show()

# Epoch at which the val_loss minimum was reached (early-stop indicator)
best_epoch = (
    diag.sort_values("val_loss")
        .groupby(["seed", "threshold"], as_index=False)
        .first()[["seed", "threshold", "epoch", "val_loss"]]
        .rename(columns={"epoch": "best_epoch", "val_loss": "best_val_loss"})
        .sort_values(["seed", "threshold"])
)
print(best_epoch.to_string(index=False))


## 3. GCN viva ou morta?

Três sinais por época:
- **`‖z‖_mean`** — norma média do embedding do nó central. Se cair para ~0 ou colapsar para um valor constante → mode collapse da GCN.
- **`Var(z)_mean`** — variância das dimensões de `z` entre amostras. Se ~0, o `z` é o mesmo para qualquer input → bypass total.
- **`‖∇gcn‖ / ‖∇lstm‖`** — razão das normas de gradiente. `< 1e-3` ⇒ ramo praticamente sem actualizações; `~0.1–1` ⇒ ramo activo.


In [ ]:
signals = [
    ("z_norm_mean",         "‖z‖ mean",          "linear"),
    ("z_var_mean",          "Var(z) mean",       "log"),
    ("grad_ratio_gcn_lstm", "‖∇gcn‖ / ‖∇lstm‖",  "log"),
]

fig, axes = plt.subplots(len(signals), len(seeds),
                         figsize=(5 * len(seeds), 3 * len(signals)),
                         sharex=True, squeeze=False)

for row, (col_name, ylabel, yscale) in enumerate(signals):
    for j, seed in enumerate(seeds):
        ax = axes[row, j]
        for t in thresholds:
            sub = diag[(diag["seed"] == seed) & (diag["threshold"] == t)].sort_values("epoch")
            if sub.empty:
                continue
            ax.plot(sub["epoch"], sub[col_name], color=colors[t], lw=1.2, label=f"th={t}")
        ax.set_ylabel(ylabel); ax.set_yscale(yscale); ax.grid(alpha=0.3)
        if row == 0:
            ax.set_title(f"Seed {seed}")
        if row == len(signals) - 1:
            ax.set_xlabel("Epoch")
        if row == 0 and j == len(seeds) - 1:
            ax.legend(fontsize=8, loc="best")

# reference lines for grad ratio
for j in range(len(seeds)):
    ax = axes[2, j]
    ax.axhline(1e-3, color="red",    ls="--", lw=0.8, alpha=0.6)
    ax.axhline(1e-1, color="orange", ls="--", lw=0.8, alpha=0.6)

fig.suptitle(f"Product {PRODUCT_ID} — GCN signal health", y=1.01)
plt.tight_layout(); plt.show()

# Final values per (seed, threshold)
final = (
    diag.sort_values("epoch")
        .groupby(["seed", "threshold"], as_index=False)
        .tail(1)[["seed", "threshold", "z_norm_mean", "z_var_mean", "grad_ratio_gcn_lstm"]]
        .sort_values(["seed", "threshold"])
)
print("Final epoch values:")
print(final.to_string(index=False))


## 4. Desempenho final — RMSE/MAE em `spearman.csv` por threshold × seed


In [ ]:
if not os.path.exists(RESULTS_PATH):
    print(f"{RESULTS_PATH} not found — skip section 4.")
else:
    res = pd.read_csv(RESULTS_PATH)
    res = res[res["product_id"] == PRODUCT_ID].copy()
    res["threshold"] = pd.to_numeric(res["threshold"], errors="coerce")

    # Baseline (ablate_z=False) for cleaner comparison; fall back to all if column missing
    if "ablate_z" in res.columns:
        base = res[res["ablate_z"].astype(str).str.lower().isin(["false", "0", "0.0"])]
    else:
        base = res

    metric_col = "rmse" if "rmse" in base.columns else base.columns[-1]
    pivot = base.pivot_table(index="seed", columns="threshold",
                             values=metric_col, aggfunc="mean")
    print(f"{metric_col.upper()} per seed × threshold (baseline):")
    print(pivot.round(4).to_string())

    summary = pivot.agg(["mean", "std"]).T.reset_index()
    print("\nAcross-seed summary:")
    print(summary.round(4).to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 4))
    for seed in pivot.index:
        ax.plot(pivot.columns, pivot.loc[seed], marker="o", label=f"seed {seed}", alpha=0.7)
    ax.errorbar(summary["threshold"], summary["mean"], yerr=summary["std"],
                color="black", lw=2, capsize=4, label="mean ± std")
    ax.set_xlabel("Threshold"); ax.set_ylabel(metric_col.upper())
    ax.set_title(f"Product {PRODUCT_ID} — {metric_col.upper()} vs threshold")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


## 5. Ablation — baseline vs `ablate_z=True` (sem GCN)

Esta secção compara o RMSE do modelo completo com o do modelo em que o `z` da GCN é zerado em forward (LSTM ainda recebe target+exog). Se `Δ = ablate − baseline > 0`, a GCN está a contribuir; se `Δ ≤ 0`, a GCN é decorativa ou prejudicial.

Para gerar a coluna `ablate_z=True` em `spearman.csv`, correr `main.py` com `ABLATE_Z = True`.


In [ ]:
if not os.path.exists(RESULTS_PATH) or "ablate_z" not in pd.read_csv(RESULTS_PATH, nrows=1).columns:
    print("Sem coluna `ablate_z` em spearman.csv — correr main.py com ABLATE_Z=True e voltar a executar.")
else:
    res = pd.read_csv(RESULTS_PATH)
    res = res[res["product_id"] == PRODUCT_ID].copy()
    res["threshold"] = pd.to_numeric(res["threshold"], errors="coerce")
    res["ablate_z"]  = res["ablate_z"].astype(str).str.lower().map(
        {"true": True, "1": True, "1.0": True, "false": False, "0": False, "0.0": False}
    )
    metric_col = "rmse" if "rmse" in res.columns else res.columns[-1]

    grp = (
        res.groupby(["threshold", "ablate_z"])[metric_col]
           .agg(["mean", "std", "count"])
           .reset_index()
    )
    print(grp.round(4).to_string(index=False))

    if grp["ablate_z"].nunique() < 2:
        print("\nFalta um dos lados (baseline ou ablation). Correr o que falta.")
    else:
        wide = grp.pivot(index="threshold", columns="ablate_z", values="mean")
        wide.columns = ["baseline", "ablate"]
        wide["delta_ablate_minus_baseline"] = wide["ablate"] - wide["baseline"]
        print("\nΔ RMSE (positivo ⇒ GCN ajuda):")
        print(wide.round(4).to_string())

        fig, ax = plt.subplots(figsize=(7, 4))
        x = np.arange(len(wide.index)); w = 0.35
        ax.bar(x - w/2, wide["baseline"], w, label="baseline (GCN on)")
        ax.bar(x + w/2, wide["ablate"],   w, label="ablate (z=0)")
        ax.set_xticks(x); ax.set_xticklabels(wide.index)
        ax.set_xlabel("Threshold"); ax.set_ylabel(metric_col.upper())
        ax.set_title(f"Product {PRODUCT_ID} — ablation comparison")
        ax.grid(alpha=0.3, axis="y"); ax.legend()
        plt.tight_layout(); plt.show()


## 6. Forecast plots (HTML)

Carrega e apresenta os plots `.html` gerados por `main.py` (`SAVE_PLOTS = True`).

- **Baseline (ablação)** — ficheiro `*_ablation.html`: corrida com `ABLATE_Z=True` (GCN zerada, LSTM pura).  
- **GCN+LSTM** — ficheiro sem sufixo `_ablation.html`: corrida com `ABLATE_Z=False`.

Configura `PLOT_SEED`, `PLOT_WINDOW`, `PLOT_STEP` e `SELECTED_THRESHOLD` abaixo.  
O plot é interactivo (Plotly): usa a legenda para mostrar/ocultar thresholds individuais.


In [ ]:
import glob
import hashlib
from IPython.display import HTML, display

# ── Plot configuration ───────────────────────────────────────────────────
PLOT_SEED          = 42       # seed used when running main.py
PLOT_WINDOW        = 15       # window_size
PLOT_STEP          = 1        # step_size
SELECTED_THRESHOLD = 0.30     # threshold to highlight in the title (informational only)
# Thresholds list used in main.py — needed to rebuild the md5 hash sub-dir name
PLOT_THRESHOLDS    = [0.30, 0.50, 0.70, 0.85, 0.95, 0.99]

# Compute the sub-dir hash exactly as main.py does
raw_str    = "_".join(map(str, PLOT_THRESHOLDS))
values_str = hashlib.md5(raw_str.encode()).hexdigest()[:8]

base_dir = os.path.join(
    SCRIPT_DIR, "grid_search_plots", f"seed_{PLOT_SEED}",
    "*",  # metric_type (similarity / distance)
    f"window_{PLOT_WINDOW}", f"step_{PLOT_STEP}",
    f"item_{PRODUCT_ID}", values_str,
)

# Locate GCN+LSTM (baseline=ablation) and full-model HTML files
ablation_pattern = os.path.join(
    base_dir, f"item_{PRODUCT_ID}_{METRIC}_seed_{PLOT_SEED}_all_configs_ablation.html"
)
fullmodel_pattern = os.path.join(
    base_dir, f"item_{PRODUCT_ID}_{METRIC}_seed_{PLOT_SEED}_all_configs.html"
)

ablation_files  = glob.glob(ablation_pattern)
fullmodel_files = glob.glob(fullmodel_pattern)

def _render_html(path: str, title: str) -> None:
    print(f"[{title}]  {os.path.relpath(path, SCRIPT_DIR)}")
    with open(path, encoding="utf-8") as fh:
        display(HTML(fh.read()))

if not ablation_files and not fullmodel_files:
    # Fallback: scan the whole plots dir for any matching html for this product/metric/seed
    fallback = glob.glob(
        os.path.join(SCRIPT_DIR, "grid_search_plots", "**",
                     f"item_{PRODUCT_ID}_{METRIC}_seed_{PLOT_SEED}_*.html"),
        recursive=True,
    )
    if fallback:
        print(f"Hash sub-dir not matched — showing all found plots ({len(fallback)} files):")
        for p in sorted(fallback):
            label = "Baseline (ablation)" if p.endswith("_ablation.html") else "GCN+LSTM"
            _render_html(p, label)
    else:
        print(
            "Nenhum ficheiro HTML encontrado. Verifica que:\n"
            "  1. SAVE_PLOTS = True em main.py\n"
            f"  2. PLOT_SEED={PLOT_SEED}, PLOT_WINDOW={PLOT_WINDOW}, PLOT_STEP={PLOT_STEP} coincidem com a corrida\n"
            f"  3. PLOT_THRESHOLDS={PLOT_THRESHOLDS} são os thresholds usados (afectam o nome da sub-pasta)\n"
            "  4. Para o plot de ablação, correr main.py com ABLATE_Z=True"
        )
else:
    # ── Baseline (ablação — ABLATE_Z=True) ──────────────────────────────
    if ablation_files:
        _render_html(ablation_files[0], "Baseline — ABLATE_Z=True (sem GCN)")
    else:
        print("Plot de ablação não encontrado — correr main.py com ABLATE_Z=True para gerá-lo.")

    # ── GCN+LSTM completo ────────────────────────────────────────────────
    if fullmodel_files:
        _render_html(
            fullmodel_files[0],
            f"GCN+LSTM — ABLATE_Z=False  (threshold seleccionado: {SELECTED_THRESHOLD})",
        )
    else:
        print("Plot GCN+LSTM completo não encontrado — correr main.py com ABLATE_Z=False.")
